# 02 — Bronze Layer
Reads raw JSON from Volumes → Writes to Bronze Delta Table
Strict schema enforcement, no inferSchema
Adds metadata: load_timestamp, batch_id, load_type

In [0]:
dbutils.widgets.text("load_type", "full", "Load Type")
dbutils.widgets.text("batch_id", "batch_fullload_2000_2026", "Batch ID")
dbutils.widgets.text("raw_file","dbfs:/Volumes/workspace/default/products/pakistan_weather_fullload.json","Raw File Path")

LOAD_TYPE = dbutils.widgets.get("load_type")
BATCH_ID  = dbutils.widgets.get("batch_id")
RAW_FILE  = dbutils.widgets.get("raw_file")  
RAW_DIR   = "dbfs:/Volumes/workspace/default/products"

print(f"Load Type : {LOAD_TYPE}")
print(f"Batch ID  : {BATCH_ID}")
print(f"Raw File  : {RAW_FILE}")

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, LongType, ArrayType,TimestampType,IntegerType
)
from pyspark.sql import functions as F
from datetime import datetime

print("Imports done!")

In [0]:
# BRONZE SCHEMA — All StringType
# Bronze = raw landing zone


hourly_schema = StructType([
    StructField("time",                              ArrayType(StringType()), True),
    StructField("temperature_2m",                    ArrayType(StringType()), True),
    StructField("relative_humidity_2m",              ArrayType(StringType()), True),
    StructField("dew_point_2m",                      ArrayType(StringType()), True),
    StructField("apparent_temperature",              ArrayType(StringType()), True),
    StructField("pressure_msl",                      ArrayType(StringType()), True),
    StructField("surface_pressure",                  ArrayType(StringType()), True),
    StructField("weather_code",                      ArrayType(StringType()), True),
    StructField("vapour_pressure_deficit",           ArrayType(StringType()), True),
    StructField("precipitation",                     ArrayType(StringType()), True),
    StructField("rain",                              ArrayType(StringType()), True),
    StructField("snowfall",                          ArrayType(StringType()), True),
    StructField("snow_depth",                        ArrayType(StringType()), True),
    StructField("wind_speed_10m",                    ArrayType(StringType()), True),
    StructField("wind_speed_100m",                   ArrayType(StringType()), True),
    StructField("wind_direction_10m",                ArrayType(StringType()), True),
    StructField("wind_direction_100m",               ArrayType(StringType()), True),
    StructField("wind_gusts_10m",                    ArrayType(StringType()), True),
    StructField("cloud_cover",                       ArrayType(StringType()), True),
    StructField("cloud_cover_low",                   ArrayType(StringType()), True),
    StructField("cloud_cover_mid",                   ArrayType(StringType()), True),
    StructField("cloud_cover_high",                  ArrayType(StringType()), True),
    StructField("shortwave_radiation",               ArrayType(StringType()), True),
    StructField("direct_radiation",                  ArrayType(StringType()), True),
    StructField("diffuse_radiation",                 ArrayType(StringType()), True),
    StructField("direct_normal_irradiance",          ArrayType(StringType()), True),
    StructField("global_tilted_irradiance",          ArrayType(StringType()), True),
    StructField("terrestrial_radiation",             ArrayType(StringType()), True),
    StructField("shortwave_radiation_instant",       ArrayType(StringType()), True),
    StructField("direct_radiation_instant",          ArrayType(StringType()), True),
    StructField("diffuse_radiation_instant",         ArrayType(StringType()), True),
    StructField("direct_normal_irradiance_instant",  ArrayType(StringType()), True),
    StructField("global_tilted_irradiance_instant",  ArrayType(StringType()), True),
    StructField("terrestrial_radiation_instant",     ArrayType(StringType()), True),
    StructField("soil_temperature_0_to_7cm",         ArrayType(StringType()), True),
    StructField("soil_temperature_7_to_28cm",        ArrayType(StringType()), True),
    StructField("soil_temperature_28_to_100cm",      ArrayType(StringType()), True),
    StructField("soil_temperature_100_to_255cm",     ArrayType(StringType()), True),
    StructField("soil_moisture_0_to_7cm",            ArrayType(StringType()), True),
    StructField("soil_moisture_7_to_28cm",           ArrayType(StringType()), True),
    StructField("soil_moisture_28_to_100cm",         ArrayType(StringType()), True),
    StructField("soil_moisture_100_to_255cm",        ArrayType(StringType()), True),
    StructField("et0_fao_evapotranspiration",        ArrayType(StringType()), True),
])

raw_schema = StructType([
    StructField("latitude",               StringType(), True),
    StructField("longitude",              StringType(), True),
    StructField("generationtime_ms",      StringType(), True),
    StructField("utc_offset_seconds",     StringType(), True),
    StructField("timezone",               StringType(), True),
    StructField("timezone_abbreviation",  StringType(), True),
    StructField("elevation",              StringType(), True),
    StructField("hourly",                 hourly_schema, True),
])

print("Bronze Schema defined!")
print("All fields: StringType — raw landing zone")
print(f"Total hourly variables: {len(hourly_schema.fields)}")

In [0]:

from pyspark.sql import functions as F
from datetime import datetime
start_time = datetime.now()

df_raw = spark.read \
    .schema(raw_schema) \
    .option("multiline", "true") \
    .json(RAW_FILE)

print(f"Raw file loaded!")
print(f"Total chunks: {df_raw.count()}")

In [0]:
df_bronze = df_raw \
    .withColumn("load_timestamp", F.current_timestamp()) \
    .withColumn("load_type",      F.lit(LOAD_TYPE)) \
    .withColumn("batch_id",       F.lit(BATCH_ID)) \
    .withColumn("source_file",    F.lit(RAW_FILE))

write_mode = "overwrite" if LOAD_TYPE == "full" else "append"
bronze_status = "Success"
bronze_error  = None
rows = 0

try:
    df_bronze.write \
        .format("delta") \
        .mode(write_mode) \
        .option("overwriteSchema", "true" if write_mode == "overwrite" else "false") \
        .option("mergeSchema", "true" if write_mode == "append" else "false") \
        .saveAsTable("pakistan_weather.bronze_weather_raw")

    end_time = datetime.now()
    rows = df_bronze.count()

    print(f"Bronze table written!")
    print(f"  Rows    : {rows:,}")
    print(f"  Columns : {len(df_bronze.columns)}")
    print(f"  Duration: {(end_time-start_time).seconds}s")
except Exception as e:
    end_time = datetime.now()
    bronze_status = "Failure"
    bronze_error  = str(e)
    print(f"BRONZE WRITE FAILED: {e}")

In [0]:

#explicit schema to handle None values cleanly
log_schema = StructType([
    StructField("layer", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("file_processed", StringType(), True),
    StructField("city", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("error_message", StringType(), True),
    StructField("batch_id", StringType(), True)
])

log_data = [{
    "layer":          "Raw-to-Bronze",
    "load_type":      LOAD_TYPE,
    "file_processed": RAW_FILE,
    "city":           "All-5-Cities",
    "start_time":     start_time,
    "end_time":       end_time,
    "status":         bronze_status,
    "rows_inserted":  rows,
    "rows_updated":   0,
    "error_message":  bronze_error,
    "batch_id":       BATCH_ID
}]

log_df = spark.createDataFrame(log_data, schema=log_schema)

log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("pakistan_weather.pipeline_execution_logs")

print("Logged successfully!")
print(f"\n{'='*45}")
print(f"BRONZE COMPLETE!")
print(f"  Rows   : {rows:,}")
print(f"  Status : {bronze_status}")
print(f"{'='*45}")

In [0]:

print("\nRow count:")
spark.sql("SELECT COUNT(*) as total_rows FROM pakistan_weather.bronze_weather_raw").show()

print("\nLog entry:")
spark.sql("SELECT * FROM pakistan_weather.pipeline_execution_logs ORDER BY log_id").show(truncate=False)

In [0]:
spark.sql("SELECT latitude, longitude, timezone, batch_id, load_timestamp FROM pakistan_weather.bronze_weather_raw ").show(truncate=False)